# 01 — Data Loading, Cleaning & Validation
### Olist — Delivery & Review Risk Analysis

**Business question:** What's driving late deliveries and poor reviews, and how much revenue is at risk because of it?

This notebook:
1. Loads all 9 raw Olist CSVs
2. Profiles each table (shape, nulls, dtypes, duplicates)
3. Cleans and type-casts key columns (dates, IDs)
4. Validates referential integrity between tables (do the joins actually hold together?)
5. Engineers the core analytical fields (delivery time, lateness, order revenue)
6. Builds a single cleaned **SQLite database** (`olist.db`) that Notebook 3 will query with pure SQL
7. Saves a cleaned, merged master table for EDA in Notebook 2


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_DIR = '../Data'
OUT_DIR = '../Outputs'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs('../Data/processed', exist_ok=True)


## 1. Load raw tables

In [2]:
orders      = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv')
items       = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
payments    = pd.read_csv(f'{DATA_DIR}/olist_order_payments_dataset.csv')
reviews     = pd.read_csv(f'{DATA_DIR}/olist_order_reviews_dataset.csv')
products    = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')
customers   = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv')
sellers     = pd.read_csv(f'{DATA_DIR}/olist_sellers_dataset.csv')
geoloc      = pd.read_csv(f'{DATA_DIR}/olist_geolocation_dataset.csv')
translation = pd.read_csv(f'{DATA_DIR}/product_category_name_translation.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments, 'reviews': reviews,
    'products': products, 'customers': customers, 'sellers': sellers,
    'geolocation': geoloc, 'category_translation': translation
}
for name, df in tables.items():
    print(f'{name:22s} shape={df.shape}')


orders                 shape=(99441, 8)
items                  shape=(112650, 7)
payments               shape=(103886, 5)
reviews                shape=(99224, 7)
products               shape=(32951, 9)
customers              shape=(99441, 5)
sellers                shape=(3095, 4)
geolocation            shape=(1000163, 5)
category_translation   shape=(71, 2)


## 2. Profile each table
Check nulls, dtypes, and duplicate keys before touching anything.

In [3]:
null_summary = {}
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        null_summary[name] = nulls.to_dict()

for name, cols in null_summary.items():
    print(f'--- {name} ---')
    for col, n in cols.items():
        pct = n / len(tables[name]) * 100
        print(f'  {col:35s} {n:6d} nulls ({pct:.1f}%)')


--- orders ---
  order_approved_at                      160 nulls (0.2%)
  order_delivered_carrier_date          1783 nulls (1.8%)
  order_delivered_customer_date         2965 nulls (3.0%)
--- reviews ---
  review_comment_title                 87656 nulls (88.3%)
  review_comment_message               58247 nulls (58.7%)
--- products ---
  product_category_name                  610 nulls (1.9%)
  product_name_lenght                    610 nulls (1.9%)
  product_description_lenght             610 nulls (1.9%)
  product_photos_qty                     610 nulls (1.9%)
  product_weight_g                         2 nulls (0.0%)
  product_length_cm                        2 nulls (0.0%)
  product_height_cm                        2 nulls (0.0%)
  product_width_cm                         2 nulls (0.0%)


In [4]:
# Duplicate primary keys check
print('Duplicate order_id in orders:      ', orders['order_id'].duplicated().sum())
print('Duplicate review_id in reviews:    ', reviews['review_id'].duplicated().sum())
print('Duplicate product_id in products:  ', products['product_id'].duplicated().sum())
print('Duplicate seller_id in sellers:    ', sellers['seller_id'].duplicated().sum())
print('Duplicate customer_id in customers:', customers['customer_id'].duplicated().sum())
print('Fully duplicated review rows:      ', reviews.duplicated().sum())


Duplicate order_id in orders:       0
Duplicate review_id in reviews:     814
Duplicate product_id in products:   0
Duplicate seller_id in sellers:     0
Duplicate customer_id in customers: 0
Fully duplicated review rows:       0


**Findings so far:**
- `orders`: missing delivery timestamps are expected — those are orders that are canceled/unavailable/still in transit, not data errors. We'll keep them but flag them.
- `reviews`: `review_comment_title` and `review_comment_message` are mostly null — that's normal, most customers leave a star rating without writing a comment. Not something to "fix", just don't rely on comment text for the core analysis.
- `products`: 610 rows missing category/description metadata, and 2 rows missing physical dimensions. Small enough (<2% of products) to drop or leave as `Unknown` rather than dropping whole rows that might still have valid order history.
- No duplicate primary keys found in the core tables — good, the joins can be trusted at the grain we expect.


## 3. Clean & type-cast

In [5]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# Fill product category gaps with 'unknown' rather than dropping rows
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Merge English category names in now so every downstream table can use one label
products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

print('Date columns cast to datetime64.')
print(orders[date_cols].dtypes)


Date columns cast to datetime64.
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## 4. Validate referential integrity
Before joining anything for real, confirm the join keys actually match across tables — this is the difference between a portfolio piece that *looks* like it uses joins and one that's actually been checked for correctness.

In [6]:
def orphan_rate(child, child_key, parent, parent_key, label):
    orphans = (~child[child_key].isin(parent[parent_key])).sum()
    pct = orphans / len(child) * 100
    print(f'{label:45s} orphans={orphans:6d} ({pct:.2f}%)')
    return orphans

orphan_rate(items, 'order_id', orders, 'order_id', 'items -> orders')
orphan_rate(items, 'product_id', products, 'product_id', 'items -> products')
orphan_rate(items, 'seller_id', sellers, 'seller_id', 'items -> sellers')
orphan_rate(payments, 'order_id', orders, 'order_id', 'payments -> orders')
orphan_rate(reviews, 'order_id', orders, 'order_id', 'reviews -> orders')
orphan_rate(orders, 'customer_id', customers, 'customer_id', 'orders -> customers')


items -> orders                               orphans=     0 (0.00%)
items -> products                             orphans=     0 (0.00%)
items -> sellers                              orphans=     0 (0.00%)
payments -> orders                            orphans=     0 (0.00%)
reviews -> orders                             orphans=     0 (0.00%)
orders -> customers                           orphans=     0 (0.00%)


np.int64(0)

All join keys resolve cleanly (0% orphans) — the star-schema-like structure (orders as the hub, joined out to items/payments/reviews/customers, and items joined out to products/sellers) is safe to use directly in SQL without needing extra dedup or fuzzy-matching logic.

## 5. Engineer core analytical fields
This is where we turn raw timestamps into the metrics the business question actually needs: delivery time, lateness, and revenue per order.

In [7]:
orders['is_delivered'] = orders['order_status'] == 'delivered'

# Delivery time in days (only meaningful for delivered orders)
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Lateness vs. the promise made at checkout (estimated delivery date)
orders['delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.total_seconds() / 86400
orders['is_late'] = orders['delay_days'] > 0

print(orders[['is_delivered', 'is_late']].sum())
print()
print(orders.loc[orders['is_delivered'], 'delivery_time_days'].describe())


is_delivered    96478
is_late          7827
dtype: int64

count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
25%          6.766204
50%         10.217477
75%         15.720182
max        209.628611
Name: delivery_time_days, dtype: float64


In [8]:
# Order-level revenue: sum of item price + freight per order (from items table)
order_revenue = items.groupby('order_id').agg(
    item_price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()
order_revenue['order_revenue'] = order_revenue['item_price'] + order_revenue['freight_value']

# Order-level review: an order can have >1 review row; keep the latest one
reviews_sorted = reviews.sort_values('review_answer_timestamp')
order_review = reviews_sorted.drop_duplicates('order_id', keep='last')[
    ['order_id', 'review_score', 'review_creation_date']
]

order_revenue.head()


,order_id,item_price,freight_value,n_items,order_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,218.04


## 6. Build the master analytical table
One row per order, joined out to customer location, revenue, and review score — this feeds Notebook 2 (EDA). SQL joins across the *raw* tables happen separately in Notebook 3.

In [9]:
master = (
    orders
    .merge(customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left')
    .merge(order_revenue, on='order_id', how='left')
    .merge(order_review, on='order_id', how='left')
)

# Bring in the dominant product category per order (the category of its highest-value item)
item_category = (
    items.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')
)
top_category_per_order = (
    item_category.sort_values('price', ascending=False)
    .drop_duplicates('order_id')[['order_id', 'product_category_name_english']]
    .rename(columns={'product_category_name_english': 'main_category'})
)
master = master.merge(top_category_per_order, on='order_id', how='left')

print(master.shape)
master.head()


(99441, 21)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time_days,delay_days,is_late,customer_city,customer_state,item_price,freight_value,n_items,order_revenue,review_score,review_creation_date,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,8.436574,-7.107488,False,sao paulo,SP,29.99,8.72,1.0,38.71,4.0,2017-10-11,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,13.782037,-5.355729,False,barreiras,BA,118.70,22.76,1.0,141.46,4.0,2018-08-08,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,9.394213,-17.245498,False,vianopolis,GO,159.90,19.22,1.0,179.12,5.0,2018-08-18,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,13.208750,-12.980069,False,sao goncalo do amarante,RN,45.00,27.20,1.0,72.20,5.0,2017-12-03,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,True,2.873877,-9.238171,False,santo andre,SP,19.90,8.72,1.0,28.62,5.0,2018-02-17,stationery


In [10]:
# How much of the master table is usable for the delivery/review analysis?
delivered = master[master['is_delivered']]
print(f"Total orders:            {len(master):,}")
print(f"Delivered orders:        {len(delivered):,} ({len(delivered)/len(master)*100:.1f}%)")
print(f"Delivered orders w/ review: {delivered['review_score'].notna().sum():,} ({delivered['review_score'].notna().mean()*100:.1f}%)")
print(f"Delivered orders w/ revenue: {delivered['order_revenue'].notna().sum():,} ({delivered['order_revenue'].notna().mean()*100:.1f}%)")


Total orders:            99,441
Delivered orders:        96,478 (97.0%)
Delivered orders w/ review: 95,832 (99.3%)
Delivered orders w/ revenue: 96,478 (100.0%)


## 7. Persist cleaned outputs
- `master` table → CSV, for fast reload in Notebook 2 (EDA)
- All cleaned raw tables → a SQLite database `olist.db`, for genuine SQL querying in Notebook 3

In [11]:
master.to_csv('../Data/processed/master_orders.csv', index=False)
print('Saved master_orders.csv')


Saved master_orders.csv


In [12]:
db_path = '../Data/processed/olist.db'
conn = sqlite3.connect(db_path)

orders.to_sql('orders', conn, if_exists='replace', index=False)
items.to_sql('order_items', conn, if_exists='replace', index=False)
payments.to_sql('order_payments', conn, if_exists='replace', index=False)
reviews.to_sql('order_reviews', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)
sellers.to_sql('sellers', conn, if_exists='replace', index=False)

# Helpful indexes for join performance
cur = conn.cursor()
for stmt in [
    'CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(order_id)',
    'CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(product_id)',
    'CREATE INDEX IF NOT EXISTS idx_items_seller ON order_items(seller_id)',
    'CREATE INDEX IF NOT EXISTS idx_payments_order ON order_payments(order_id)',
    'CREATE INDEX IF NOT EXISTS idx_reviews_order ON order_reviews(order_id)',
    'CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)',
]:
    cur.execute(stmt)
conn.commit()

tbls = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
conn.close()
print('SQLite DB written to', db_path)
tbls


SQLite DB written to ../Data/processed/olist.db


,name
0,orders
1,order_items
2,order_payments
3,order_reviews
4,products
5,customers
6,sellers
